# Fine-Tune YOLOv8n on Sentinel-1 SAR with AIS Annotations

**Maritime Edge AI Platform — Phase 0 → Phase 1 Transition**

This notebook fine-tunes a YOLOv8n detector (pre-trained on iVision-MRSSD simulated SAR)
on **real Sentinel-1 GRD data** with **3,321 AIS vessel annotations** from Global Fishing Watch.

---
### Quick Start
1. Upload `maritime_dataset.zip` to your Google Drive
2. Run the cells below in order
3. The fine-tuned model will be saved to your Drive
---

## 1. Install Dependencies

In [ ]:
# Install ultralytics and dependencies
!pip install -q ultralytics matplotlib pillow numpy tqdm

import ultralytics
ultralytics.__version__

In [ ]:
import zipfile
import json
import os
from pathlib import Path
from ultralytics import YOLO

print("Dependencies loaded")

## 2. Mount Google Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Config — UPDATE THIS PATH to your ZIP location
ZIP_PATH = "/content/drive/MyDrive/maritime_dataset.zip"
EXTRACT_PATH = "/content/maritime_dataset"

print(f"Looking for ZIP at: {ZIP_PATH}")

In [ ]:
if not os.path.exists(ZIP_PATH):
    # Fallback: upload ZIP directly via Colab file browser
    from google.colab import files
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    ZIP_PATH = f"/content/{zip_name}"
    print(f"Uploaded: {zip_name}")

# Extract
os.makedirs(EXTRACT_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_PATH)

# Find dataset.yaml
yaml_path = None
for f in Path(EXTRACT_PATH).rglob("*.yaml"):
    if 'dataset' in f.name.lower():
        yaml_path = f
        break

if yaml_path is None:
    raise FileNotFoundError("dataset.yaml not found in extracted archive")

print(f"Dataset extracted to: {EXTRACT_PATH}")
print(f"Config file: {yaml_path}")

# Display dataset summary if available
summary_path = yaml_path.parent / "dataset_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(f"\nDataset: {summary['total_images']} images, {summary['total_boxes']} boxes")
    print(f"Split: train={summary['split']['train']}, val={summary['split']['val']}, test={summary['split']['test']}")
    print(f"Classes: {summary['classes']}")
else:
    print(f"\nDataset YAML contents:")
    print(yaml_path.read_text())

## 3. Inspect Dataset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

# Count files per split
for split in ['train', 'val', 'test']:
    img_dir = yaml_path.parent / split / 'images'
    lbl_dir = yaml_path.parent / split / 'labels'
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob('*.png')))
        n_lbls = len(list(lbl_dir.glob('*.txt')))
        print(f"{split}: {n_imgs} images, {n_lbls} labels")

# Show sample tiles with GT boxes
train_img_dir = yaml_path.parent / 'train' / 'images'
train_lbl_dir = yaml_path.parent / 'train' / 'labels'
sample_imgs = sorted(train_img_dir.glob('*.png'))[:3]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img_path in zip(axes, sample_imgs):
    img = np.array(Image.open(img_path))
    ax.imshow(img, cmap='gray')
    
    # Draw GT labels
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - w/2) * 512
                y1 = (cy - h/2) * 512
                rect = patches.Rectangle((x1, y1), w*512, h*512,
                    linewidth=2, edgecolor='lime', facecolor='none')
                ax.add_patch(rect)
    
    ax.set_title(img_path.stem[-20:], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()
print("Dataset looks good, ready to train")

## 4. Download Base Model

We start from YOLOv8n pre-trained on COCO. The domain adaptation
from COCO natural images to SAR will happen via fine-tuning.

> Optionally, if you have the MRSSD-pretrained model,
> upload it to Drive and use it as the starting point instead.

In [ ]:
# Option A: Start from YOLOv8n COCO (recommended)
model = YOLO("yolov8n.pt")
print("Base model loaded: YOLOv8n (COCO)")

# Option B (uncomment): Start from MRSSD-pretrained weights
# MRSSD_PATH = "/content/drive/MyDrive/yolov8n_mrssd_v1.pt"
# model = YOLO(MRSSD_PATH)
# print("Base model loaded: YOLOv8n (MRSSD)")

## 5. Fine-Tuning Configuration

Hyperparameters optimized for SAR domain adaptation:

In [ ]:
TRAIN_CONFIG = {
    "data": str(yaml_path),
    "epochs": 100,
    "patience": 20,          # Early stopping if no improvement
    "batch": 16,
    "imgsz": 512,            # Match our 512x512 tile size
    "lr0": 0.001,            # Initial learning rate
    "lrf": 0.01,             # Final LR factor (cosine scheduler)
    "optimizer": "AdamW",
    "cos_lr": True,
    "warmup_epochs": 3,
    "warmup_momentum": 0.8,
    "weight_decay": 0.0005,
    "momentum": 0.937,
    "close_mosaic": 10,      # Disable mosaic augmentation at end
    "augment": True,
    # SAR-specific augmentation tuning
    "hsv_h": 0.0,            # No hue shift (grayscale)
    "hsv_s": 0.0,            # No saturation shift
    "hsv_v": 0.2,            # Mild brightness variation
    "degrees": 0.0,          # No rotation (SAR geometry matters)
    "translate": 0.1,        # Mild translation
    "scale": 0.2,            # Mild scaling
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,           # No vertical flip
    "fliplr": 0.5,           # Horizontal flip OK
    "project": "/content/runs",
    "name": "maritime_finetune",
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
    "seed": 42,
}

print("Training configuration ready")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

## 6. Run Fine-Tuning

Expected duration: 2-4 hours on a T4 GPU.
Checkpoints are saved every 10 epochs. Early stopping at 20 epochs without improvement.

In [ ]:
print("Starting fine-tuning...")
print(f"  Dataset: {TRAIN_CONFIG['data']}")
print(f"  Epochs:  {TRAIN_CONFIG['epochs']}")
print(f"  Batch:   {TRAIN_CONFIG['batch']}")
print(f"  Img sz:  {TRAIN_CONFIG['imgsz']}")

results = model.train(**TRAIN_CONFIG)

print("Fine-tuning complete!")

## 7. Evaluate on Test Set

In [ ]:
# Load best checkpoint
best_path = "/content/runs/maritime_finetune/weights/best.pt"
best_model = YOLO(best_path)

# Evaluate on test split
metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=512,
    batch=16,
    conf=0.25,
    iou=0.5,
)

print()
print("=" * 60)
print("  TEST SET EVALUATION")
print("=" * 60)
print(f"  mAP@0.5:     {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision:    {metrics.box.mp:.4f}")
print(f"  Recall:       {metrics.box.mr:.4f}")
print("=" * 60)

# Apply Phase 0 decision criteria
if metrics.box.map50 > 0.70:
    print("GO — mAP@0.5 > 0.70, proceed to Phase 1")
elif metrics.box.map50 > 0.50:
    print("MARGINAL — mAP@0.5 between 0.50 and 0.70")
    print("  Consider more epochs or additional data")
elif metrics.box.map50 > 0.30:
    print("BELOW THRESHOLD — mAP@0.5 < 0.50")
    print("  Consider: MRSSD pretrained weights, more data, or architecture change")
else:
    print("FAILED — mAP@0.5 < 0.30, model not learning")
    print("  Check: dataset integrity, augmentation config, learning rate")

## 8. Sample Predictions

Red boxes = model predictions  |  Green dashed = ground truth

In [ ]:
# Run inference on test set samples
test_img_dir = yaml_path.parent / 'test' / 'images'
test_lbl_dir = yaml_path.parent / 'test' / 'labels'
test_imgs = sorted(test_img_dir.glob('*.png'))[:6]

results_list = best_model(
    [str(p) for p in test_imgs],
    conf=0.25,
    imgsz=512,
)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path, result in zip(axes.flatten(), test_imgs, results_list):
    # Use result.plot() to get image with predictions drawn
    plotted = result.plot()
    ax.imshow(plotted)
    
    # Draw ground truth boxes (green dashed)
    lbl_path = test_lbl_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - w/2) * 512
                y1 = (cy - h/2) * 512
                rect = patches.Rectangle(
                    (x1, y1), w*512, h*512,
                    linewidth=1.5, edgecolor='lime', facecolor='none', linestyle='--'
                )
                ax.add_patch(rect)
    
    ax.set_title(f"{img_path.stem[-20:]}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. Export Fine-Tuned Model

Export to ONNX for edge deployment (Raspberry Pi / Jetson Nano).

In [ ]:
# Export to ONNX (FP32)
export_path = best_model.export(format="onnx", imgsz=512)
print(f"Model exported: {export_path}")

# Export to ONNX (INT8 quantized)
int8_path = best_model.export(
    format="onnx",
    imgsz=512,
    int8=True,
    data=str(yaml_path),
)
print(f"INT8 model exported: {int8_path}")

# Copy models to Google Drive for persistence
import shutil
drive_models = "/content/drive/MyDrive/maritime_models"
os.makedirs(drive_models, exist_ok=True)

for src in [export_path, int8_path]:
    dst = os.path.join(drive_models, os.path.basename(src))
    shutil.copy2(src, dst)
    print(f"Saved to Drive: {dst}")

## 10. Summary

| Metric | Value |
|--------|-------|
| Base model | YOLOv8n (COCO pretrained) |
| Training images | ~1,235 (80%) |
| Validation images | ~154 (10%) |
| Test images | ~155 (10%) |
| Total annotations | 3,321 |
| Tile size | 512×512 |
| Pipeline | D (Sigma0+Lee+Log+HistEq) |
| mAP@0.5 (test) | *fill after training* |
| mAP@0.5:0.95 (test) | *fill after training* |

---

**Next steps after fine-tuning:**
1. Copy ONNX models to the project `shared/models/` directory
2. Re-run `benchmark_pipeline.py` on all 4 pipelines
3. If mAP > 0.70 → Phase 1 microservice deployment
4. Deploy to edge (Raspberry Pi / Jetson Nano)